# 🌽 Maize Disease Classification - Complete Guide

**Project:** Maize Disease Classification with Deep Learning  
**Approach:** Two-stage transfer learning (PlantVillage → Maize_in_Field)  
**Target:** Local laptop development (GPU/CPU)

---

## 📋 What this notebook covers

| Stage | Dataset | Goal |
|-------|---------|------|
| **Stage 1** | PlantVillage | Learn disease features on clean data (~85-95% accuracy) |
| **Stage 2** | Maize_in_Field | Fine-tune for real-world South African field conditions |

## 📁 Expected folder structure
```
Maize_disease_classification_model/
├── notebooks/          ← You are here
├── data/
│   ├── plantvillage/   ← Downloaded in Step 0
│   ├── plantvillage_maize_only/  ← Created in Step 2
│   ├── maize-in-field-dataset.zip  ← Already downloaded
│   └── maize_in_field/  ← Extracted in Step 0
├── models/
├── logs/
└── results/
```

⚠️ **Run cells in order from top to bottom. Do not skip cells!**

---
# 📥 STEP 0: Dataset Setup

Run this section **once** to get both datasets ready.

### 0A: Extract Maize_in_Field (already downloaded)
You have `data/maize-in-field-dataset.zip`. Run the cell below to extract it.

In [ ]:
import os
import zipfile

zip_path = '../data/maize-in-field-dataset.zip'
extract_path = '../data/'

if os.path.exists(zip_path):
    print(f'Found zip: {zip_path} ({os.path.getsize(zip_path)/(1024**3):.1f} GB)')
    print('Extracting... (this may take a few minutes)')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print('✅ Maize_in_Field extracted!')
    
    # Show what was extracted
    for item in os.listdir(extract_path):
        print(f'  {item}')
else:
    print('⚠️ Zip not found at:', zip_path)
    print('Place maize-in-field-dataset.zip in the data/ folder')

### 0B: Download PlantVillage Dataset from Kaggle

**One-time setup — follow these steps:**

1. Go to [https://www.kaggle.com/settings](https://www.kaggle.com/settings)
2. Scroll to **API** → click **Create New Token**
3. This downloads `kaggle.json` to your Downloads folder
4. Move `kaggle.json` to: `C:\Users\kmosw\.kaggle\kaggle.json`
   - Create the `.kaggle` folder if it doesn't exist
5. Then run the cell below

In [ ]:
import os
import subprocess

# Check if kaggle.json exists
kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')
if not os.path.exists(kaggle_json):
    print('❌ kaggle.json not found!')
    print(f'Expected location: {kaggle_json}')
    print('\nSteps to fix:')
    print('  1. Go to https://www.kaggle.com/settings')
    print('  2. API → Create New Token → download kaggle.json')
    print(f'  3. Move it to: {kaggle_json}')
else:
    print('✅ kaggle.json found!')

# Check if PlantVillage already downloaded
pv_path = '../data/plantvillage'
if os.path.exists(pv_path):
    print(f'\n✅ PlantVillage already exists at {pv_path}')
    print('Skip to STEP 1 — no need to re-download')
else:
    print(f'\n📥 PlantVillage not yet downloaded. Run the next cell.')

In [ ]:
# ⚠️ Only run this if PlantVillage is NOT already downloaded
# This downloads ~1.5 GB — may take 10-30 minutes depending on internet speed

import subprocess
import os

os.makedirs('../data/plantvillage', exist_ok=True)

print('Downloading PlantVillage dataset from Kaggle...')
print('This will take 10-30 minutes. Please wait...')

result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', 'abdallahalidev/plantvillage-dataset',
     '--unzip', '-p', '../data/plantvillage'],
    capture_output=True, text=True
)

if result.returncode == 0:
    print('✅ PlantVillage downloaded successfully!')
    print(f'Location: ../data/plantvillage/')
else:
    print('❌ Download failed:')
    print(result.stderr)
    print('\nAlternative: Download manually from:')
    print('  https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset')
    print('  Extract to: ../data/plantvillage/')

---
# ═══════════════════════════════════════
# 🌱 STAGE 1: PlantVillage Baseline Training
# ═══════════════════════════════════════

**Goal:** Train a ResNet50 model on clean PlantVillage data to learn disease features.  
**Expected accuracy:** 85–95% on validation set  
**Expected time:** 1–6 hours depending on hardware

---
## 📦 STEP 1: Import Everything

In [ ]:
import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

# TensorFlow / Keras core
import tensorflow as tf
from tensorflow.keras import Model, layers, Sequential
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger, TensorBoard
)
from tensorflow.keras.optimizers import Adam

# Metrics
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {tf.config.list_physical_devices("GPU")}')

if tf.config.list_physical_devices('GPU'):
    print('✅ GPU detected — training will be much faster!')
else:
    print('⚠️ No GPU detected — training on CPU (will take longer)')
    print('   Tip: Reduce epochs to 10-20 for a quick test run')

---
## 🗂️ STEP 2: Create Maize-Only Dataset

PlantVillage has many crops (tomato, potato, etc.). We extract **only maize/corn** classes.

⚠️ **Run this once only.** Skip if `../data/plantvillage_maize_only/` already exists.

In [ ]:
# Check what maize folders exist in PlantVillage
source_dir = '../data/plantvillage/color'  # ← Update path if yours is different

if not os.path.exists(source_dir):
    # Try alternative paths
    for alt in ['../data/plantvillage/Plant_leave_diseases_dataset_with_augmentation',
                '../data/plantvillage/plantvillage dataset/color',
                '../data/plantvillage']:
        if os.path.exists(alt):
            source_dir = alt
            print(f'Found PlantVillage at: {source_dir}')
            break
    else:
        print('❌ PlantVillage not found! Run Step 0B first.')
        raise FileNotFoundError('PlantVillage dataset not found.')

all_folders = os.listdir(source_dir)
maize_folders = [f for f in all_folders if 'corn' in f.lower() or 'maize' in f.lower()]

print(f'Total folders in PlantVillage: {len(all_folders)}')
print(f'Maize folders found: {len(maize_folders)}')
print('\nMaize classes:')
for folder in maize_folders:
    count = len(os.listdir(os.path.join(source_dir, folder)))
    print(f'  - {folder} ({count} images)')

In [ ]:
# Create clean maize-only dataset (run once)
dest_dir = '../data/plantvillage_maize_only'

if os.path.exists(dest_dir):
    print(f'✅ Maize-only dataset already exists at {dest_dir}')
    print('   Skip this cell or delete the folder to recreate')
else:
    os.makedirs(dest_dir, exist_ok=True)
    
    for folder in maize_folders:
        src = os.path.join(source_dir, folder)
        dst = os.path.join(dest_dir, folder)
        
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            img_count = len(os.listdir(dst))
            print(f'✅ Copied {folder} ({img_count} images)')

    print(f'\n🌽 Maize-only dataset created with {len(maize_folders)} classes at {dest_dir}')

---
## 📂 STEP 3: Load Training Data

In [ ]:
data_dir = '../data/plantvillage_maize_only'

# Load training data
train_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='training',
    seed=123,
    image_size=(224, 224),
    batch_size=16  # ← Reduce to 8 or 4 if you get memory errors
)

# Load validation data
val_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=(224, 224),
    batch_size=16
)

# Get class names
class_names = train_ds.class_names
print(f'\n🎯 Training on {len(class_names)} maize classes:')
for i, name in enumerate(class_names, 1):
    print(f'  {i}. {name}')

---
## 📊 STEP 4: Quick Data Check

In [ ]:
# Count images
train_count = train_ds.cardinality().numpy() * 16
val_count = val_ds.cardinality().numpy() * 16
print(f'Training: ~{train_count} images | Validation: ~{val_count} images')

# Visualize samples
plt.figure(figsize=(12, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]].replace('Corn_(maize)___', ''), fontsize=9)
        plt.axis('off')
plt.suptitle('PlantVillage - Sample Training Images', fontsize=14)
plt.tight_layout()
plt.savefig('../results/stage1_sample_images.png', dpi=100)
plt.show()
print('✅ Sample images saved to results/')

---
## 🔧 STEP 5: Add Data Augmentation

In [ ]:
# Data augmentation (training only — makes model more robust)
data_augmentation = Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
])

# Apply augmentation to training set
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))

# Optimize for speed (cache + prefetch)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print('✅ Augmentation applied to training data')
print('   Validation data: no augmentation (for accurate evaluation)')

---
## 🏗️ STEP 6: Build the Model

We use **ResNet50** pretrained on ImageNet as the base. The classification head is trained from scratch.

In [ ]:
# Load pre-trained ResNet50 (downloads weights automatically first time)
print('Loading ResNet50 pretrained weights...')
base_model = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze base — only train classification head

# Build complete model
preprocess = layers.Rescaling(1./255)  # Normalize pixel values to [0, 1]

inputs = Input(shape=(224, 224, 3))
x = preprocess(inputs)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)  # Prevent overfitting
outputs = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs, outputs)

# Compile
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f'\n✅ Model built!')
print(f'   Output classes: {len(class_names)}')
print(f'   Total parameters: {model.count_params():,}')
# model.summary()  # Uncomment for full architecture details

---
## 🚀 STEP 7: Setup Training Callbacks

In [ ]:
# Create output directories
os.makedirs('../models', exist_ok=True)
os.makedirs('../logs', exist_ok=True)
os.makedirs('../results', exist_ok=True)

callbacks = [
    # Save best model automatically
    ModelCheckpoint(
        filepath='../models/stage1_best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    
    # Stop early if no improvement for 10 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate when stuck
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    
    # Log metrics to CSV (survives notebook restart!)
    CSVLogger('../logs/stage1_training_log.csv'),
    
    # TensorBoard visualization
    TensorBoard(log_dir='../logs/stage1', histogram_freq=1)
]

print('✅ Callbacks configured!')
print(f'   Model checkpoint → {os.path.abspath("../models/stage1_best_model.h5")}')
print(f'   Training log → {os.path.abspath("../logs/stage1_training_log.csv")}')
print()
print('💡 TIP: To view TensorBoard, open a terminal and run:')
print('   tensorboard --logdir=../logs/stage1')
print('   Then open: http://localhost:6006')

---
## 🎯 STEP 8: TRAIN! (Stage 1)

⏱️ **Expected time:** 1–6 hours depending on hardware (GPU/CPU)  
💻 Make sure your laptop is plugged in and sleep mode is disabled  
📊 The best model is automatically saved — you can safely close the notebook after training

| Hardware | Expected Time |
|----------|---------------|
| GPU (RTX 3070+) | 30–60 min |
| GPU (GTX 1660) | 1–2 hours |
| CPU (i7/i9) | 3–5 hours |
| CPU (i5) | 5–8 hours |

In [ ]:
print('=' * 60)
print('STAGE 1: PLANTVILLAGE BASELINE TRAINING')
print('=' * 60)
print(f'Classes: {class_names}')
print(f'Epochs: 50 (early stopping may stop earlier)')
print('=' * 60)
print()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,   # ← Reduce to 10 for a quick test run
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Stage 1 training complete!')

---
## 🔄 RESUMING AFTER RESTART

If you closed the notebook after training, run this cell to reload everything before continuing with Steps 9–12.

In [ ]:
# ── Only run this if you closed the notebook after training ──
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf, pandas as pd
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import confusion_matrix, classification_report

# Reload validation data
data_dir = '../data/plantvillage_maize_only'
val_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=(224, 224),
    batch_size=16
).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
class_names = val_ds.class_names

# Load saved model
model = load_model('../models/stage1_best_model.h5')
print('✅ Model reloaded!')
print(f'Classes: {class_names}')

---
## 📈 STEP 9: Plot Training Results

In [ ]:
# Load training history from CSV (works even after closing notebook)
history_df = pd.read_csv('../logs/stage1_training_log.csv')
print(f'Training completed: {len(history_df)} epochs')
print(f'\nFinal Results:')
print(f'  Training Accuracy:   {history_df["accuracy"].iloc[-1]:.4f}')
print(f'  Validation Accuracy: {history_df["val_accuracy"].iloc[-1]:.4f}')
print(f'  Best Val Accuracy:   {history_df["val_accuracy"].max():.4f}')
print(f'  Training Loss:       {history_df["loss"].iloc[-1]:.4f}')
print(f'  Validation Loss:     {history_df["val_loss"].iloc[-1]:.4f}')

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(history_df['accuracy'], label='Training', linewidth=2)
ax1.plot(history_df['val_accuracy'], label='Validation', linewidth=2)
ax1.set_title('Stage 1: Model Accuracy', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])

ax2.plot(history_df['loss'], label='Training', linewidth=2)
ax2.plot(history_df['val_loss'], label='Validation', linewidth=2)
ax2.set_title('Stage 1: Model Loss', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/stage1_training_curves.png', dpi=100)
plt.show()
print('✅ Training curves saved to results/')

---
## 🎯 STEP 10: Evaluate Model on Validation Set

In [ ]:
# Generate predictions on validation set
print('Generating predictions...')
y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
short_names = [n.replace('Corn_(maize)___', '') for n in class_names]

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_names, yticklabels=short_names)
plt.title('Stage 1: Confusion Matrix (PlantVillage)', fontsize=14)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../results/stage1_confusion_matrix.png', dpi=100)
plt.show()

# Classification Report
print('\n📊 Classification Report:\n')
print(classification_report(y_true, y_pred, target_names=short_names))

# Store for Stage 2 comparison
stage1_report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
stage1_val_acc = stage1_report['accuracy']
print(f'Stage 1 Validation Accuracy: {stage1_val_acc:.2%}')

---
## 👀 STEP 11: Visualize Predictions

In [ ]:
def show_predictions(dataset, model, class_names, num_images=9, title='Predictions'):
    """Visualize model predictions on sample images"""
    plt.figure(figsize=(15, 15))
    short_names = [n.replace('Corn_(maize)___', '') for n in class_names]
    
    for images, labels in dataset.take(1):
        predictions = model.predict(images, verbose=0)
        
        for i in range(min(num_images, len(images))):
            ax = plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].numpy().astype('uint8'))
            
            true_idx = labels[i].numpy()
            pred_idx = np.argmax(predictions[i])
            confidence = np.max(predictions[i]) * 100
            
            true_name = short_names[true_idx]
            pred_name = short_names[pred_idx]
            color = 'green' if true_idx == pred_idx else 'red'
            
            plt.title(
                f'True: {true_name}\nPred: {pred_name} ({confidence:.1f}%)',
                color=color, fontsize=9
            )
            plt.axis('off')
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    return plt

fig = show_predictions(val_ds, model, class_names, title='Stage 1: Sample Predictions (PlantVillage)')
plt.savefig('../results/stage1_predictions.png', dpi=100)
plt.show()
print('✅ Prediction images saved to results/')

---
## 💾 STEP 12: Save Stage 1 Model

In [ ]:
# Save final Stage 1 model (needed for Stage 2!)
model.save('../models/stage1_final_model.h5')
print('✅ Stage 1 final model saved!')

# Save training summary
history_df = pd.read_csv('../logs/stage1_training_log.csv')
stage1_summary = {
    'stage': 1,
    'dataset': 'PlantVillage',
    'total_epochs': len(history_df),
    'final_train_accuracy': float(history_df['accuracy'].iloc[-1]),
    'final_val_accuracy': float(history_df['val_accuracy'].iloc[-1]),
    'best_val_accuracy': float(history_df['val_accuracy'].max()),
    'classes': class_names
}

with open('../models/stage1_summary.json', 'w') as f:
    json.dump(stage1_summary, f, indent=4)

print('\n📁 Stage 1 files saved:')
print(f'   Final model: ../models/stage1_final_model.h5')
print(f'   Best model:  ../models/stage1_best_model.h5')
print(f'   Summary:     ../models/stage1_summary.json')
print(f'   Training log:../logs/stage1_training_log.csv')
print()
print(f'🎯 Stage 1 Validation Accuracy: {stage1_summary["best_val_accuracy"]:.2%}')
print()
if stage1_summary['best_val_accuracy'] >= 0.80:
    print('✅ Good result! Ready for Stage 2 fine-tuning.')
else:
    print('⚠️ Accuracy below 80%. Consider training longer or adjusting hyperparameters.')

---
# ═══════════════════════════════════════
# 🌍 STAGE 2: Real-World Fine-Tuning
# ═══════════════════════════════════════

**Goal:** Adapt the Stage 1 model to real South African field conditions.  
**Dataset:** Maize_in_Field (already in your `data/` folder)  
**Expected improvement:** +10–20% accuracy on real-world images  
**Expected time:** 2–5 hours

⚠️ **Prerequisites:** Stage 1 must be complete and `stage1_final_model.h5` saved in `models/`

---
## 🗂️ S2 STEP 1: Check Maize_in_Field Dataset

In [ ]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import json
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras import layers, Sequential
from sklearn.metrics import confusion_matrix, classification_report

# Check if dataset exists - try common extraction paths
possible_paths = [
    '../data/maize_in_field',
    '../data/maize-in-field-dataset',
    '../data/Maize_in_Field',
]

maize_field_path = None
for path in possible_paths:
    if os.path.exists(path):
        # Check it has subdirectories (class folders)
        items = os.listdir(path)
        dirs = [i for i in items if os.path.isdir(os.path.join(path, i))]
        if dirs:
            maize_field_path = path
            break

if maize_field_path is None:
    # List data folder contents to help diagnose
    print('⚠️ Maize_in_Field dataset not found in expected locations.')
    print('\nContents of data/ folder:')
    for item in os.listdir('../data'):
        print(f'  {item}')
    print('\n→ Run Step 0A to extract the zip, or update the path below')
    maize_field_path = '../data/maize_in_field'  # ← Update this if needed
else:
    print(f'✅ Found Maize_in_Field dataset at: {maize_field_path}')
    folders = [f for f in os.listdir(maize_field_path)
               if os.path.isdir(os.path.join(maize_field_path, f))]
    print(f'\nClass distribution:')
    for folder in sorted(folders):
        fpath = os.path.join(maize_field_path, folder)
        count = len([f for f in os.listdir(fpath)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f'  {folder}: {count} images')

---
## 🏷️ S2 STEP 2: Align Class Names

Maize_in_Field uses different folder names than PlantVillage. We create a standardized copy.

In [ ]:
# PlantVillage class names (from Stage 1)
plantvillage_classes = [
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight',
    'Corn_(maize)___healthy'
]

# Check actual Maize_in_Field folder names
field_classes = sorted([
    f for f in os.listdir(maize_field_path)
    if os.path.isdir(os.path.join(maize_field_path, f))
])

print('PlantVillage classes:')
for i, name in enumerate(plantvillage_classes):
    print(f'  {i}: {name}')

print('\nMaize_in_Field classes:')
for i, name in enumerate(field_classes):
    print(f'  {i}: {name}')

# Class mapping — adjust if your folder names are different
# The keys are Maize_in_Field folder names, values are PlantVillage names
class_mapping = {
    'Gray_Leaf_Spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Common_Rust':    'Corn_(maize)___Common_rust_',
    'Blight':         'Corn_(maize)___Northern_Leaf_Blight',
    'Healthy':        'Corn_(maize)___healthy'
}

print('\nClass mapping (Field → PlantVillage):')
for k, v in class_mapping.items():
    found = '✅' if k in field_classes else '❌ NOT FOUND'
    print(f'  {found} {k} → {v}')

# If any class is missing, update class_mapping above to match your actual folder names

In [ ]:
# Create standardized copy of Maize_in_Field with PlantVillage-compatible names
standardized_path = '../data/maize_in_field_standardized'

if os.path.exists(standardized_path):
    print(f'✅ Standardized dataset already exists at {standardized_path}')
else:
    os.makedirs(standardized_path, exist_ok=True)
    print('Creating standardized dataset...')
    
    for field_name, pv_name in class_mapping.items():
        src = os.path.join(maize_field_path, field_name)
        dst = os.path.join(standardized_path, pv_name)
        
        if os.path.exists(src):
            if not os.path.exists(dst):
                shutil.copytree(src, dst)
                count = len(os.listdir(dst))
                print(f'  ✅ {field_name} → {pv_name} ({count} images)')
        else:
            print(f'  ❌ {src} not found! Update class_mapping above.')
    
    print(f'\n✅ Standardized dataset created!')

field_data_dir = standardized_path

---
## 📊 S2 STEP 3: Load Maize_in_Field Data

In [ ]:
print('Loading Maize_in_Field dataset...')

# 70/30 split for fine-tuning
train_field_ds = image_dataset_from_directory(
    field_data_dir,
    validation_split=0.3,
    subset='training',
    seed=123,
    image_size=(224, 224),
    batch_size=8,  # Smaller batch for fine-tuning
    shuffle=True
)

val_field_ds = image_dataset_from_directory(
    field_data_dir,
    validation_split=0.3,
    subset='validation',
    seed=123,
    image_size=(224, 224),
    batch_size=8
)

field_class_names = train_field_ds.class_names
print(f'Classes: {field_class_names}')
print(f'Training batches: {train_field_ds.cardinality().numpy()}')
print(f'Validation batches: {val_field_ds.cardinality().numpy()}')

# Verify class names match PlantVillage (critical for fine-tuning!)
if field_class_names == plantvillage_classes:
    print('\n✅ Class names match PlantVillage — fine-tuning will work correctly!')
else:
    print('\n⚠️ Warning: Class names do NOT match PlantVillage!')
    print('Expected:', plantvillage_classes)
    print('Got:', field_class_names)
    print('→ Go back and fix the class_mapping in S2 Step 2')

---
## 🔧 S2 STEP 4: Apply Light Augmentation

We use **less** augmentation than Stage 1 — real field images already have natural variation.

In [ ]:
# Light augmentation for real-world data
field_augmentation = Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),   # Less than Stage 1 (0.2)
    layers.RandomZoom(0.05),       # Less than Stage 1 (0.2)
])

# Apply to training data only
train_field_ds = train_field_ds.map(
    lambda x, y: (field_augmentation(x, training=True), y)
)

# Optimize performance
AUTOTUNE = tf.data.AUTOTUNE
train_field_ds = train_field_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_field_ds = val_field_ds.cache().prefetch(buffer_size=AUTOTUNE)

print('✅ Light augmentation applied (less aggressive than Stage 1)')

---
## 📊 S2 STEP 5: Visualize Field Data Samples

In [ ]:
short_field_names = [n.replace('Corn_(maize)___', '') for n in field_class_names]

plt.figure(figsize=(12, 12))
for images, labels in train_field_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(short_field_names[labels[i]], fontsize=9)
        plt.axis('off')
plt.suptitle('Maize_in_Field — Real-World Samples', fontsize=14)
plt.tight_layout()
plt.savefig('../results/stage2_field_samples.png', dpi=100)
plt.show()
print('Notice: More variation in lighting, angles, backgrounds vs PlantVillage')

---
## 🎯 S2 STEP 6: Baseline — How does Stage 1 model do on field data?

Before fine-tuning, we measure the performance gap. Expect a **drop of 15–25%** — this is normal!

In [ ]:
# Load Stage 1 model
print('Loading Stage 1 model...')
stage1_model = load_model('../models/stage1_final_model.h5')

print('\n' + '=' * 60)
print('STAGE 1 MODEL ON REAL-WORLD DATA (Before Fine-tuning)')
print('=' * 60)

y_true = []
y_pred = []

for images, labels in val_field_ds:
    predictions = stage1_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=short_field_names, yticklabels=short_field_names)
plt.title('Stage 1 Model on Real-World Data (Before Fine-tuning)', fontsize=13)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../results/stage1_on_field_confusion.png', dpi=100)
plt.show()

# Report
stage1_field_report = classification_report(
    y_true, y_pred, target_names=field_class_names, output_dict=True
)
print(classification_report(y_true, y_pred, target_names=short_field_names))

stage1_field_acc = stage1_field_report['accuracy']
print(f'📊 Stage 1 on Real-World Data: {stage1_field_acc:.2%}')
print('\n⚠️ Performance drop is EXPECTED — real-world data is harder!')
print('   Fine-tuning in Stage 2 will close this gap.')

---
## 🔓 S2 STEP 7: Prepare Model for Fine-Tuning

We unfreeze the **top 20 layers** of ResNet50 so they can adapt to field conditions.  
We use a **very low learning rate** (100x lower than Stage 1) to avoid catastrophic forgetting.

In [ ]:
from tensorflow.keras.optimizers import Adam

# Start from Stage 1 model
model = load_model('../models/stage1_final_model.h5')

# Get the ResNet50 base model (layer index 2 in our architecture)
base_model = model.layers[2]
print(f'Base model: {base_model.name}')
print(f'Total layers in base: {len(base_model.layers)}')

# Unfreeze top 20 layers of ResNet50
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'\nTrainable base layers: {trainable_count}/{len(base_model.layers)}')

# Recompile with VERY LOW learning rate (crucial!)
model.compile(
    optimizer=Adam(learning_rate=0.00001),  # 100x lower than Stage 1
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f'\n✅ Model ready for fine-tuning')
print(f'   Learning rate: 0.00001 (very low — prevents catastrophic forgetting)')
print(f'   Total trainable parameters: {sum(tf.keras.backend.count_params(p) for p in model.trainable_weights):,}')

---
## 🚀 S2 STEP 8: Fine-Tune on Real-World Data

⏱️ **Expected time:** 2–5 hours  
💻 Keep laptop plugged in, sleep mode disabled

In [ ]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger, TensorBoard
)

os.makedirs('../logs/stage2', exist_ok=True)

stage2_callbacks = [
    ModelCheckpoint(
        filepath='../models/stage2_best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-8,
        verbose=1
    ),
    CSVLogger('../logs/stage2_training_log.csv'),
    TensorBoard(log_dir='../logs/stage2', histogram_freq=1)
]

print('=' * 60)
print('STAGE 2: FINE-TUNING ON REAL-WORLD DATA')
print('=' * 60)
print(f'Dataset: Maize_in_Field')
print(f'Epochs: 30 (early stopping may end sooner)')
print(f'Learning Rate: 0.00001')
print(f'Unfrozen base layers: 20')
print('=' * 60)
print()

history_stage2 = model.fit(
    train_field_ds,
    validation_data=val_field_ds,
    epochs=30,
    callbacks=stage2_callbacks,
    verbose=1
)

print('\n✅ Stage 2 fine-tuning complete!')

---
## 📈 S2 STEP 9: Evaluate Stage 2 Results

In [ ]:
history_df = pd.read_csv('../logs/stage2_training_log.csv')

print('\n' + '=' * 60)
print('STAGE 2 RESULTS')
print('=' * 60)
print(f'Total epochs trained: {len(history_df)}')
print(f'Final Training Accuracy:   {history_df["accuracy"].iloc[-1]:.4f}')
print(f'Final Validation Accuracy: {history_df["val_accuracy"].iloc[-1]:.4f}')
print(f'Best Validation Accuracy:  {history_df["val_accuracy"].max():.4f}')

best_val_acc = history_df['val_accuracy'].max()

print(f'\n📊 Progression:')
print(f'  Stage 1 on PlantVillage:   82%+ (expected)')
print(f'  Stage 1 on field (before): {stage1_field_acc:.1%}')
print(f'  Stage 2 on field (after):  {best_val_acc:.1%}')
print(f'  Improvement: {(best_val_acc - stage1_field_acc)*100:+.1f}%')

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(history_df['accuracy'], label='Training', linewidth=2)
ax1.plot(history_df['val_accuracy'], label='Validation', linewidth=2)
ax1.axhline(y=stage1_field_acc, color='r', linestyle='--',
            label=f'Stage 1 Baseline ({stage1_field_acc:.1%})', alpha=0.7)
ax1.set_title('Stage 2: Fine-tuning Accuracy', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_df['loss'], label='Training', linewidth=2)
ax2.plot(history_df['val_loss'], label='Validation', linewidth=2)
ax2.set_title('Stage 2: Loss', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/stage2_training_curves.png', dpi=100)
plt.show()
print('✅ Stage 2 training curves saved')

---
## 🎯 S2 STEP 10: Detailed Evaluation

In [ ]:
# Load best Stage 2 model
stage2_model = load_model('../models/stage2_best_model.h5')

y_true = []
y_pred = []

print('Generating predictions on real-world validation set...')
for images, labels in val_field_ds:
    predictions = stage2_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=short_field_names, yticklabels=short_field_names)
plt.title('Stage 2: Fine-tuned Model on Real-World Data', fontsize=13)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../results/stage2_confusion_matrix.png', dpi=100)
plt.show()

# Classification Report
print('\n' + '=' * 60)
print('STAGE 2: CLASSIFICATION REPORT')
print('=' * 60)

stage2_report = classification_report(
    y_true, y_pred, target_names=field_class_names, output_dict=True
)
print(classification_report(y_true, y_pred, target_names=short_field_names))

# Per-class improvement table
print('\n' + '=' * 60)
print('PER-CLASS IMPROVEMENT (Stage 1 → Stage 2)')
print('=' * 60)

comparison_data = []
for class_name, short_name in zip(field_class_names, short_field_names):
    before_f1 = stage1_field_report[class_name]['f1-score']
    after_f1 = stage2_report[class_name]['f1-score']
    comparison_data.append({
        'Disease': short_name,
        'Before F1': f'{before_f1:.3f}',
        'After F1': f'{after_f1:.3f}',
        'Change': f'{(after_f1 - before_f1):+.3f}'
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

---
## 👀 S2 STEP 11: Visualize Predictions on Real Field Images

In [ ]:
plt.figure(figsize=(15, 15))

for images, labels in val_field_ds.take(1):
    predictions = stage2_model.predict(images, verbose=0)
    
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        
        true_idx = labels[i].numpy()
        pred_idx = np.argmax(predictions[i])
        confidence = np.max(predictions[i]) * 100
        
        true_short = short_field_names[true_idx]
        pred_short = short_field_names[pred_idx]
        color = 'green' if true_idx == pred_idx else 'red'
        
        plt.title(
            f'True: {true_short}\nPred: {pred_short}\n({confidence:.1f}%)',
            color=color, fontsize=9
        )
        plt.axis('off')

plt.suptitle('Stage 2: Predictions on Real-World Field Images', fontsize=14)
plt.tight_layout()
plt.savefig('../results/stage2_predictions.png', dpi=100)
plt.show()
print('✅ Stage 2 predictions saved')

---
## 💾 S2 STEP 12: Save Final Model & Export

In [ ]:
# Copy best model to final name
shutil.copy('../models/stage2_best_model.h5', '../models/maize_final_model.h5')

# Convert to TensorFlow Lite for mobile deployment
print('Converting to TensorFlow Lite...')
converter = tf.lite.TFLiteConverter.from_keras_model(stage2_model)
tflite_model = converter.convert()

with open('../models/maize_final_model.tflite', 'wb') as f:
    f.write(tflite_model)

tflite_size = len(tflite_model) / (1024 * 1024)
print(f'✅ TFLite model size: {tflite_size:.2f} MB')

# Save final summary JSON
history_df = pd.read_csv('../logs/stage2_training_log.csv')
final_summary = {
    'stage1': {'dataset': 'PlantVillage', 'description': 'Clean lab images'},
    'stage1_on_field': {
        'dataset': 'Maize_in_Field',
        'accuracy': float(stage1_field_acc),
        'description': 'Stage 1 on real-world data before fine-tuning'
    },
    'stage2': {
        'dataset': 'Maize_in_Field',
        'accuracy': float(best_val_acc),
        'epochs_trained': len(history_df),
        'description': 'Fine-tuned model on real-world data'
    },
    'improvement': {
        'absolute_pct': float((best_val_acc - stage1_field_acc) * 100),
    },
    'per_class_f1_stage2': {
        cn.replace('Corn_(maize)___', ''): float(stage2_report[cn]['f1-score'])
        for cn in field_class_names
    },
    'class_names': field_class_names,
    'tflite_size_mb': round(tflite_size, 2)
}

with open('../results/final_model_summary.json', 'w') as f:
    json.dump(final_summary, f, indent=4)

print('\n' + '=' * 60)
print('📁 ALL FILES SAVED')
print('=' * 60)
print(f'  models/maize_final_model.h5        ← Full Keras model')
print(f'  models/maize_final_model.tflite    ← Mobile deployment')
print(f'  models/stage1_final_model.h5       ← Stage 1 model')
print(f'  logs/stage1_training_log.csv       ← Stage 1 history')
print(f'  logs/stage2_training_log.csv       ← Stage 2 history')
print(f'  results/final_model_summary.json   ← Full summary')
print(f'  results/*.png                      ← All charts')
print()
print('=' * 60)
print('🏆 FINAL RESULTS')
print('=' * 60)
print(f'  Stage 1 on field (before): {stage1_field_acc:.1%}')
print(f'  Stage 2 on field (after):  {best_val_acc:.1%}')
print(f'  Improvement:               {(best_val_acc - stage1_field_acc)*100:+.1f}%')
print()
if best_val_acc >= 0.80:
    print('🌟 Excellent! Model is production-ready.')
elif best_val_acc >= 0.75:
    print('✅ Good! Model is deployable for testing.')
else:
    print('⚠️ Accuracy below target. Consider more epochs or adjusting learning rate.')

---
## 🔍 BONUS: Predict on a Single New Image

In [ ]:
def predict_single_image(image_path, model=None, class_names=None, threshold=0.70):
    """Quick prediction on any image with confidence threshold"""
    if model is None:
        model = load_model('../models/maize_final_model.h5')
    if class_names is None:
        with open('../results/final_model_summary.json') as f:
            class_names = json.load(f)['class_names']
    
    short_names = [n.replace('Corn_(maize)___', '') for n in class_names]
    
    # Load and preprocess image
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)
    
    # Predict
    predictions = model.predict(img_array, verbose=0)
    pred_idx = np.argmax(predictions)
    confidence = np.max(predictions) * 100
    
    # Display
    plt.figure(figsize=(6, 5))
    plt.imshow(img)
    color = 'green' if confidence >= threshold * 100 else 'orange'
    plt.title(
        f'Prediction: {short_names[pred_idx]}\nConfidence: {confidence:.1f}%' +
        ('' if confidence >= threshold * 100 else '\n⚠️ Low confidence — inspect manually'),
        color=color, fontsize=12
    )
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    return short_names[pred_idx], confidence

# Usage:
# predicted_class, confidence = predict_single_image('path/to/your/maize_image.jpg')
print('✅ predict_single_image() function ready!')
print('Usage: predict_single_image("path/to/image.jpg")')

---
## 📺 View TensorBoard

To view live training curves in TensorBoard, open a terminal/command prompt and run:

```bash
cd C:\Users\kmosw\Documents\Projects\Maize_disease_classification_model
tensorboard --logdir=logs
```

Then open your browser to: **http://localhost:6006**

---
**Good luck! 🌽🚀**  
*If something goes wrong, check `Maize_Notebook_Execution_Guide.md` for troubleshooting tips.*